# D&D Beyond Homebrew Sync

Sync homebrew spells from Google Sheets to D&D Beyond.

## Setup Instructions

1. **Get your D&D Beyond cookies**:
   - Open D&D Beyond in your browser
   - Log in to your account
   - Open Developer Tools (F12)
   - Go to Network tab
   - Visit any D&D Beyond page
   - Find the request, right-click → Copy → Copy as cURL
   - Extract the Cookie header value

2. **Get security tokens**:
   - Go to https://www.dndbeyond.com/homebrew/creations/create-spell/create
   - Open Developer Tools → Network tab
   - Fill out the form (don't submit yet)
   - Right-click → Inspect on the form
   - Find the hidden `security-token` and `authenticity-token` fields

3. **Configure below** with your cookies and tokens

## Important Notes

- **Security tokens expire**: You'll need to refresh them periodically
- **Rate limiting**: D&D Beyond may throttle requests
- **Master subscription**: Some features require a Master-tier subscription


<input id="field-security-token" name="security-token" type="hidden" value="bf877cec19a71926736b4fc2826c3d9f">

## 1. Setup and Imports

In [1]:
import pandas as pd
import requests
import json
import time
import re
import os
from typing import Dict, List, Optional
from dotenv import load_dotenv
from datetime import datetime
from urllib.parse import urlencode
from bs4 import BeautifulSoup

from DNDBeyond.core.Helpers.DnDBeyondAPI import DnDBeyondAPI

print("✓ Imports loaded")

✓ Imports loaded


## 2. D&D Beyond Authentication

Configure your D&D Beyond cookies and security tokens.

**Note**: The sync will automatically track DDB spell IDs in your Google Sheets by adding them to a "DDB" column.

In [2]:
# ============================================
# D&D Beyond Configuration
# ============================================

load_dotenv()

DDB_BASE_URL = os.getenv("DDB_BASE_URL")
DDB_COOKIES = os.getenv("DDB_COOKIES")
DDB_SECURITY_TOKEN = os.getenv("DDB_SECURITY_TOKEN")
DDB_AUTHENTICITY_TOKEN = os.getenv("DDB_AUTHENTICITY_TOKEN")
REQUEST_VERIFICATION_TOKEN = os.getenv("REQUEST_VERIFICATION_TOKEN")
DDB_USER_ID = os.getenv("DDB_USER_ID")
DDB_USERNAME = os.getenv("DDB_USERNAME")

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if DDB_COOKIES:
    session.headers.update({
        "Cookie": DDB_COOKIES,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Referer": f"{DDB_BASE_URL}/homebrew/creations/create-spell/create",
        "Origin": DDB_BASE_URL,
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1"
    })
    print("✓ Using cookie authentication")
    print(f"✓ Base URL: {DDB_BASE_URL}")
    print(f"✓ User: {DDB_USERNAME} (ID: {DDB_USER_ID})")
    
    if DDB_SECURITY_TOKEN and DDB_AUTHENTICITY_TOKEN:
        print("✓ Security tokens configured")
    else:
        print("⚠️  Security tokens not set - you'll need these to create spells!")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set DDB_COOKIES with your browser session cookies")

✓ Using cookie authentication
✓ Base URL: https://www.dndbeyond.com
✓ User: Rockoteque (ID: 110746825)
✓ Security tokens configured


## 3. Load Spell Data from Google Sheets

In [3]:
import FiveETools.core.fantasy.spells
from FiveETools.core.fantasy.sources import source as expected_source

# Load fantasy spells
print("Loading spells from Google Sheets...")
spells = FiveETools.core.fantasy.spells.spells_list

print(f"✓ Loaded {len(spells)} spells")

# DIAGNOSTIC: Check what's filtering out the spells
print("\n" + "=" * 60)
print("DIAGNOSTIC: Source Filter Analysis")
print("=" * 60)
print(f"Expected source value: '{expected_source}'")

SPELLS_GID = "625265890"  # Fantasy spells sheet GID
from FiveETools.core.Helpers.gsheets_client import fantasy_sheets

df_spells = fantasy_sheets.get_sheet(SPELLS_GID)
print(f"\nTotal rows in spreadsheet: {len(df_spells)}")

# Check Source column
if 'Source' in df_spells.columns:
    source_counts = df_spells['Source'].value_counts()
    print(f"\nSource values found in spreadsheet:")
    for src, count in source_counts.items():
        if pd.notna(src):
            match_indicator = " ✓ MATCH" if src == expected_source else ""
            print(f"  '{src}': {count} spells{match_indicator}")
    
    # Count spells that would match
    matching_spells = df_spells[
        (df_spells['Spell Name'].notna()) & 
        (df_spells['Spell Name'].astype(str).str.strip() != "") &
        (df_spells['Source'] == expected_source)
    ]
    print(f"\nSpells matching filter (Source == '{expected_source}'): {len(matching_spells)}")
else:
    print("\n⚠️  WARNING: 'Source' column not found in spreadsheet!")
    print(f"   Available columns: {list(df_spells.columns[:10])}...")

print("=" * 60)

if len(spells) == 0:
    print("\n❌ NO SPELLS LOADED - This is why sync didn't update anything!")
    print("   Fix: Update the 'Source' column in your spreadsheet to match the expected value")
    print(f"   Or change FiveETools/core/fantasy/sources.py to match your spreadsheet's Source values")
else:
    print(f"\nSample spell:")
    if spells:
        sample = spells[0]
        print(f"  Name: {sample.get('name', 'N/A')}")
        print(f"  Level: {sample.get('level', 'N/A')}")
        print(f"  School: {sample.get('school', 'N/A')}")

# Also load the raw DataFrame to check for existing DDB IDs
print("\nLoading spreadsheet data for DDB ID tracking...")
df_spells = fantasy_sheets.get_sheet(SPELLS_GID)
print(f"✓ Loaded spreadsheet with {len(df_spells)} rows")

# Ensure DDB column exists
print("\nEnsuring 'DDB' column exists in spreadsheet...")
try:
    ddb_col_idx = fantasy_sheets.ensure_column_exists(SPELLS_GID, "DDB")
    print(f"✓ 'DDB' column exists at index {ddb_col_idx}")
    
    # Check if any spells already have DDB IDs
    df_spells = fantasy_sheets.get_sheet(SPELLS_GID)  # Reload after ensuring column
    if 'DDB' in df_spells.columns:
        existing_ddb_ids = df_spells[df_spells['DDB'].notna()]['DDB'].tolist()
        print(f"✓ Found {len(existing_ddb_ids)} spells already synced to D&D Beyond")
    else:
        print("  'DDB' column created")
except Exception as e:
    print(f"⚠️  Could not ensure DDB column: {e}")
    print("  You may need to check your Google Sheets credentials")
    print("  Continuing without spreadsheet tracking...")

Loading spells from Google Sheets...
✓ Loaded 406 spells

DIAGNOSTIC: Source Filter Analysis
Expected source value: 'ORIO'

Total rows in spreadsheet: 406

Source values found in spreadsheet:
  'ORIO': 406 spells ✓ MATCH

Spells matching filter (Source == 'ORIO'): 406

Sample spell:
  Name: Abhaaldrac’s Unmaking Descent
  Level: 9
  School: V

Loading spreadsheet data for DDB ID tracking...
✓ Loaded spreadsheet with 406 rows

Ensuring 'DDB' column exists in spreadsheet...
✓ 'DDB' column exists at index 86
✓ Found 391 spells already synced to D&D Beyond


## 4. D&D Beyond API Helper

Helper class for interacting with D&D Beyond's homebrew creation endpoints.

In [4]:


ddb_api = DnDBeyondAPI(session, DDB_SECURITY_TOKEN, DDB_AUTHENTICITY_TOKEN, REQUEST_VERIFICATION_TOKEN)
print("✓ D&D Beyond API helper initialized")

✓ D&D Beyond API helper initialized


In [5]:
from DNDBeyond.core.Helpers import convert_spell_to_ddb

print("✓ D&D Beyond converter loaded")

✓ D&D Beyond converter loaded


## 5.5 Duplicate Checking (HTML Parsing)

The `get_user_spells()` method now uses HTML parsing to fetch existing homebrew spells.

**How it works:**

D&D Beyond uses Server-Side Rendering (SSR) and embeds spell data in the HTML response rather than providing a separate API endpoint. The method:

1. Fetches `/my-creations?filter-type=1118725998` (spell type filter)
2. Parses the HTML to find all elements with `class="list-row-homebrew-creation-Spell"`
3. Extracts spell IDs and names from the `data-slug` attributes
   - Format: `data-slug="3135831-fingerbang"`
   - Parsed to: `{"id": "3135831", "name": "Fingerbang"}`

**Filter Status Options:**
- `filter-status=1` - Published spells only
- `filter-status=2` - Draft spells only
- Use both parameters to fetch all spells

You can test the duplicate checking in the next cell.

In [6]:
# Test fetching existing spells with HTML parsing
print("Testing get_user_spells() method...\n")

try:
    existing_spells = ddb_api.get_user_spells()
    
    if existing_spells:
        print(f"✓ Successfully parsed {len(existing_spells)} existing homebrew spells\n")
        print("Found spells:")
        for i, spell in enumerate(existing_spells, 1):
            spell_name = spell.get('name', 'Unknown')
            spell_id = spell.get('id', 'Unknown')
            print(f"{i:2d}. {spell_name} (ID: {spell_id})")
            print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{spell_id}")
        
        print(f"\n✓ Duplicate checking is ready to use!")
        print(f"  When syncing, spells with matching names will be skipped.")
    else:
        print("⚠️  No spells found")
        print("\nPossible reasons:")
        print("1. You don't have any published homebrew spells yet")
        print("2. Your session cookies may have expired")
        print("3. The HTML structure may have changed")
        print("\nTo include draft spells, update the params in get_user_spells():")
        print('  params = {"filter-type": "1118725998", "filter-status": "2"}')

except Exception as e:
    print(f"✗ Error fetching spells: {e}")
    print("\nThis could mean:")
    print("1. Session cookies have expired - update DDB_COOKIES")
    print("2. D&D Beyond's HTML structure changed")
    print("3. Network or authentication issue")
    print("\nYou can still sync without duplicate checking.")
    import traceback
    traceback.print_exc()

Testing get_user_spells() method...

✓ Successfully parsed 20 existing homebrew spells

Found spells:
 1. abhaaldracs unmaking descent (ID: 3137383)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137383
 2. aegis of rebounding force (ID: 3137437)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137437
 3. alerrims solitude (ID: 3137187)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137187
 4. amphibious downpour (ID: 3137188)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137188
 5. architects collapse (ID: 3137192)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137192
 6. arouse (ID: 3137360)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137360
 7. ashen breath (ID: 3137195)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137195
 8. barb of living ruin (ID: 3137196)
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137196
 9. barriers of officialism (ID: 3137456)
    URL

## 6. Sync Spells to D&D Beyond

**Configuration Options:**
- `DRY_RUN`: Set to `False` to actually create/update spells (default: `False`)
- `BATCH_SIZE`: Set to `None` to sync all spells, or a number to limit (e.g., `10` for testing)
- `DELAY`: Seconds between requests to avoid rate limiting (default: `2`)
- `VERBOSE`: Detailed logging for debugging (default: `True`)
- `SKIP_EXISTING`: Set to `True` to skip spells that already exist (default: `True`)
  - When `True`: Spells with DDB IDs in spreadsheet or found on D&D Beyond will be skipped (if `UPDATE_EXISTING=False`)
  - When `False`: Will attempt to create spells even if they already exist (may create duplicates or fail)
- `UPDATE_EXISTING`: **NEW!** Set to `True` to UPDATE spells when DDB ID exists (default: `True`)
  - When `True` and `SKIP_EXISTING=False`: Spells with DDB IDs will be UPDATED instead of skipped
  - This is useful when you rename spells in Google Sheets - the sync will update the spell name on D&D Beyond
  - When `False` or `SKIP_EXISTING=True`: Spells with DDB IDs will be skipped (old behavior)
- `UPDATE_ADDITIONAL_FIELDS`: Set to `True` to update AOE, save, attack fields after creation (default: `True`)

**Typical Usage Scenarios:**

1. **Initial sync (create all spells)**:
   - `SKIP_EXISTING=True`, `UPDATE_EXISTING=False`
   - Skips spells that already exist (based on DDB ID in spreadsheet)

2. **Update renamed spells**:
   - `SKIP_EXISTING=False`, `UPDATE_EXISTING=True`
   - Updates all spells that have DDB IDs (handles name changes)
   - Creates new spells that don't have DDB IDs

3. **Dry run to test**:
   - `DRY_RUN=True`
   - Shows what would happen without making changes

**Error Logging:**
When errors occur, the sync will log:
- HTTP status codes and response details
- Exception types and full tracebacks
- Converted spell data for debugging
- All details are saved to `sync_log_YYYYMMDD_HHMMSS.json`

**⚠️ Important**: 
- D&D Beyond doesn't have a bulk API - this creates/updates spells one by one
- Rate limiting: D&D Beyond may throttle requests (adjust `DELAY` if needed)
- Security tokens expire: You may need to refresh them mid-sync
- Check the spreadsheet after sync - DDB IDs will be automatically recorded

In [7]:
# ============================================
# SYNC SPELLS (SIMPLIFIED)
# ============================================

from DNDBeyond.core.Helpers.sync import SpellSyncManager
from DNDBeyond.core.Helpers import convert_spell_to_ddb, normalize_ddb_id

# Configuration
DRY_RUN = False
BATCH_SIZE = None  # None for all, or number to limit
SKIP_EXISTING = False
UPDATE_EXISTING = True
UPDATE_ADDITIONAL_FIELDS = True
VERBOSE = True

# Create sync manager
sync_manager = SpellSyncManager(
    ddb_api=ddb_api,
    fantasy_sheets=fantasy_sheets,
    spells_gid=SPELLS_GID,
    delay=1
)

# Run sync
results = sync_manager.sync_base_spells(
    spells=spells,
    converter_func=convert_spell_to_ddb,
    normalize_ddb_id_func=normalize_ddb_id,
    dry_run=DRY_RUN,
    batch_size=BATCH_SIZE,
    skip_existing=SKIP_EXISTING,
    update_existing=UPDATE_EXISTING,
    update_additional_fields=UPDATE_ADDITIONAL_FIELDS,
    verbose=VERBOSE
)

print(f"\n✓ Sync complete: {results['created']} created, {results['updated']} updated, {results['errors']} errors")

⚠️  LIVE MODE - Spells WILL be created/updated!
Start time: 2026-02-15 17:30:02
Batch size: ALL SPELLS
Skip existing: NO
Update existing: YES

Total spells to process: 406

✓ Found 391 spells with DDB IDs in spreadsheet
✓ Found 20 existing homebrew spells

[1/406] Processing: Abhaaldrac’s Unmaking Descent
  → Converting spell to D&D Beyond format...
  → Updating existing spell (ID: 3137383)
  ✗ Failed to update spell
[2/406] Processing: Aegis of Rebounding Force
  → Converting spell to D&D Beyond format...
  → Updating existing spell (ID: 3137437)
  ✓ Updated spell successfully
[3/406] Processing: Alerrim's Solitude
  → Converting spell to D&D Beyond format...
  → Updating existing spell (ID: 3137187)
  ✗ Failed to update spell
[4/406] Processing: Amphibious Downpour
  → Converting spell to D&D Beyond format...
  → Updating existing spell (ID: 3137188)
  ✓ Updated spell successfully
[5/406] Processing: Architect’s Collapse
  → Converting spell to D&D Beyond format...
  → Updating exist

In [9]:
# ============================================
# SYNC CONDITIONS, MODIFIERS & HIGHER LEVELS (SIMPLIFIED)
# ============================================

from DNDBeyond.core.Helpers.sync import SpellSyncManager
from DNDBeyond.core.Helpers import (
    extract_spell_conditions,
    extract_spell_modifiers,
    extract_spell_scaling,
    parse_dice_scaling,
    normalize_ddb_id
)

# Configuration
SYNC_CONDITIONS = True
SYNC_MODIFIERS = True
SYNC_HIGHER_LEVELS = True
ONLY_NEW_SPELLS = False
DRY_RUN_EXTRAS = False
CLEAN_UPDATE = True

# Create sync manager
sync_manager = SpellSyncManager(
    ddb_api=ddb_api,
    fantasy_sheets=fantasy_sheets,
    spells_gid=SPELLS_GID,
    delay=1
)

# Run extras sync
stats = sync_manager.sync_extras(
    spells=spells,
    extract_conditions_func=extract_spell_conditions,
    extract_modifiers_func=extract_spell_modifiers,
    extract_scaling_func=extract_spell_scaling,
    parse_scaling_func=parse_dice_scaling,
    normalize_ddb_id_func=normalize_ddb_id,
    sync_conditions=SYNC_CONDITIONS,
    sync_modifiers=SYNC_MODIFIERS,
    sync_higher_levels=SYNC_HIGHER_LEVELS,
    only_new_spells=ONLY_NEW_SPELLS,
    dry_run=DRY_RUN_EXTRAS,
    clean_update=CLEAN_UPDATE
)

print(f"\n✓ Extras sync complete: {stats['modifiers_created']} modifiers, {stats['conditions_created']} conditions, {stats['higher_levels_created']} higher levels")

SYNC CONDITIONS, MODIFIERS & HIGHER LEVELS
⚠️  LIVE MODE - Changes WILL be made!
Sync Conditions: YES
Sync Modifiers: YES
Sync Higher Levels: YES
Clean Update: YES

Found 406 spells with DDB IDs

[1/406] Processing: Abhaaldrac’s Unmaking Descent (ID: 3137383)
  → Adding 2 modifier(s)
    ✗ Failed to create modifier: HTTP 200: OK | 

<!DOCTYPE html>
<!--[if lt IE 7 ]> <html lang="en" class="no-js ie6"> <![endif]-->
<!--[if IE 7 ]>    <html lang="en" class="no-js ie7"> <![endif]-->
<!--[if IE 8 ]>    <html lang="en" class="n
    ✗ Failed to create modifier: HTTP 200: OK | 

<!DOCTYPE html>
<!--[if lt IE 7 ]> <html lang="en" class="no-js ie6"> <![endif]-->
<!--[if IE 7 ]>    <html lang="en" class="no-js ie7"> <![endif]-->
<!--[if IE 8 ]>    <html lang="en" class="n
[2/406] Processing: Aegis of Rebounding Force (ID: 3137437)
  → Adding 2 modifier(s)
    ✗ Failed to create modifier: HTTP 200: OK | 

<!DOCTYPE html>
<!--[if lt IE 7 ]> <html lang="en" class="no-js ie6"> <![endif]-->
<!--[if I

## 6.5 Sync Conditions, Modifiers, and Higher Levels (with Update Support)

**This cell adds/updates Conditions, Modifiers, and "At Higher Levels" data to existing spells.**

After creating/updating the base spell information, this cell:
1. **Adds/Updates Conditions** - Creates condition effects from the "Condition" CSV column
2. **Adds/Updates Modifiers** - Creates modifier effects from the "Modifiers JSON" CSV column (supports multiple modifiers per spell)
3. **Adds/Updates "At Higher Levels"** - Creates scaling data from the "Scaling" CSV column (if structured)

**Configuration:**
- `SYNC_CONDITIONS`: Set to `True` to sync conditions (default: `True`)
- `SYNC_MODIFIERS`: Set to `True` to sync modifiers (default: `True`)
- `SYNC_HIGHER_LEVELS`: Set to `True` to sync higher level scaling (default: `False`)
- `ONLY_NEW_SPELLS`: Set to `True` to only process newly created spells (default: `False`)
- `DRY_RUN_EXTRAS`: Set to `True` to preview without making changes (default: `False`)
- **`CLEAN_UPDATE`**: Set to `True` to delete existing before creating new ones (default: `True`)
  - **Recommended: `True`** - Ensures clean updates without duplicates
  - When `True`: Fetches existing modifiers/conditions/scaling, deletes them, then creates new ones
  - When `False`: Just creates new ones (may result in duplicates if run multiple times)

**Update Handling:**

With `CLEAN_UPDATE = True` (default), the sync will:
1. **Fetch** existing modifiers/conditions/scaling from D&D Beyond
2. **Delete** all existing entries
3. **Create** new entries from CSV data

This prevents duplicates and ensures the D&D Beyond spell matches your CSV data exactly.

**Example Output:**
```
[1/406] Processing: Battle-Fury Ascension (ID: 3137528)
  → Found existing: 2 modifiers, 0 conditions, 0 higher levels
  → Adding 2 modifier(s)
    ✓ Created modifier (AC)
    ✓ Created modifier (Damage)

✓ Deleted: 2 modifiers, 0 conditions, 0 higher levels
✓ Created: 2 modifiers, 0 conditions, 0 higher levels
```

**Modifier Extraction:**

To populate the "Modifiers JSON" column, run the LLM extraction script:
```bash
# Extract from Google Sheets directly (recommended)
poetry run python DNDBeyond/scripts/extract_modifiers_llm.py --google-sheets

# Or from local CSV
poetry run python DNDBeyond/scripts/extract_modifiers_llm.py
```

The LLM will extract modifiers like:
- AC bonuses (e.g., "+2 to AC")
- Attack roll bonuses (e.g., "+1 to hit")
- Damage bonuses (e.g., "1d6 extra damage")
- Saving throw bonuses (e.g., "+2 to Dexterity saves")
- Speed modifiers (e.g., "+10 feet movement")
- Temporary HP (e.g., "2d4 temporary hit points")

**Multiple Modifiers:**

If a spell has multiple modifiers (stored as JSON array), this cell will create ALL of them:
```python
# Example: Spell with AC bonus AND damage bonus
[
  {"type": "1", "fixed_value": "2", "details": "+2 to AC"},
  {"type": "3", "dice_count": "1", "dice_type": "6", "details": "1d6 extra damage"}
]
```

**Scaling Format Options:**

For structured dice scaling (creates actual scaling entries):
```
Format: "level:dice,level:dice,..."
Example: "3:2d6,5:3d6,7:4d6,9:5d6"
```

For text descriptions only, leave as free text - it will be included in the spell description.

## 7. Batch Delete Spells from D&D Beyond

**⚠️ DANGER ZONE - This will permanently delete spells!**

Use this section to bulk delete spells from D&D Beyond. This is useful for:
- Cleaning up duplicate spells created by mistake
- Removing old versions after renaming spells
- Starting fresh with a clean slate

**How it works:**
1. Fetches all your existing spells from D&D Beyond
2. Allows you to filter spells to delete (by name pattern, ID range, etc.)
3. Deletes the selected spells one by one
4. Optionally clears DDB IDs from the spreadsheet

**Configuration:**
- `DRY_RUN`: Set to `False` to actually delete (always test with `True` first!)
- `DELETE_FILTER`: Function to filter which spells to delete
- `CLEAR_SPREADSHEET_IDS`: Set to `True` to also remove DDB IDs from spreadsheet

In [ ]:
# ============================================
# BATCH DELETE SPELLS
# ============================================

SKIP_DELETE = True  # Set to False to enable delete functionality

if SKIP_DELETE:
    print("=" * 60)
    print("⚠️  DELETE SECTION SKIPPED (SKIP_DELETE = True)")
    print("=" * 60)
    print("Set SKIP_DELETE = False in the cell above to enable delete functionality")
    print("=" * 60)
else:
    DRY_RUN = True  # ALWAYS test with True first!
    DELAY = 1  # Seconds between deletions
    VERBOSE = True  # Detailed logging
    CLEAR_SPREADSHEET_IDS = True  # Set to True to also clear DDB column in spreadsheet
    
    # Define a filter function to select which spells to delete
    # Examples:
    #   - Delete all spells: lambda spell: True
    #   - Delete spells with specific IDs: lambda spell: spell["id"] in ["3136976", "3136977"]
    #   - Delete spells by name pattern: lambda spell: "old" in spell["name"].lower()
    #   - Delete spells with ID > X: lambda spell: int(spell["id"]) > 3136000
    
    DELETE_FILTER = lambda spell: False  # Default: delete nothing (safe)
    
    # Example filters (uncomment one to use):
    # DELETE_FILTER = lambda spell: spell["id"] in ["3136976", "3136977"]  # Delete specific IDs
    # DELETE_FILTER = lambda spell: int(spell["id"]) > 3136900  # Delete recent spells
    # DELETE_FILTER = lambda spell: "ashen breath" in spell["name"].lower()  # Delete by name
    
    print("=" * 60)
    print("⚠️  DANGER: BATCH DELETE SPELLS")
    print("=" * 60)
    if DRY_RUN:
        print("DRY RUN - No spells will be deleted")
    else:
        print("⚠️  LIVE MODE - Spells WILL BE PERMANENTLY DELETED!")
    print("=" * 60)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Delay between deletions: {DELAY}s")
    print(f"Clear spreadsheet IDs: {'YES' if CLEAR_SPREADSHEET_IDS else 'NO'}")
    print()
    
    # Fetch all existing spells from D&D Beyond
    print("Fetching all homebrew spells from D&D Beyond...")
    try:
        all_spells = ddb_api.get_user_spells()
        print(f"✓ Found {len(all_spells)} existing homebrew spells\n")
    except Exception as e:
        print(f"✗ Error fetching spells: {e}")
        print("Cannot proceed with deletion")
        raise
    
    # Apply filter to select spells to delete
    spells_to_delete = [spell for spell in all_spells if DELETE_FILTER(spell)]
    
    print(f"Filter selected {len(spells_to_delete)} spells for deletion:")
    print()
    if len(spells_to_delete) == 0:
        print("✓ No spells selected for deletion")
        print("  Update DELETE_FILTER to select spells")
    else:
        for i, spell in enumerate(spells_to_delete, 1):
            print(f"{i:3d}. {spell['name']:50s} (ID: {spell['id']})")
        print()
        
        # Safety check
        if not DRY_RUN:
            print("⚠️  WARNING: You are about to PERMANENTLY DELETE these spells!")
        
        results = {"deleted": 0, "errors": 0, "details": []}
        spreadsheet_clears = []
        
        for i, spell in enumerate(spells_to_delete, 1):
            spell_name = spell['name']
            spell_id = spell['id']
            print(f"[{i}/{len(spells_to_delete)}] Processing: {spell_name} (ID: {spell_id})")
            
            if DRY_RUN:
                print(f"  → DRY RUN: Would delete spell")
                print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{spell_id}")
                results["deleted"] += 1
            else:
                try:
                    if VERBOSE:
                        print(f"  → Deleting spell from D&D Beyond...")
                    deleted = ddb_api.delete_spell(spell_id, spell_name)
                    if deleted:
                        print(f"  ✓ Deleted successfully")
                        results["deleted"] += 1
                        if CLEAR_SPREADSHEET_IDS:
                            spreadsheet_clears.append({'match_value': spell_name, 'update_column': 'DDB', 'update_value': ''})
                    else:
                        print(f"  ✗ Failed to delete")
                        results["errors"] += 1
                    if i < len(spells_to_delete):
                        time.sleep(DELAY)
                except Exception as e:
                    print(f"  ✗ Unexpected error: {e}")
                    results["errors"] += 1
        
        # Clear spreadsheet IDs if requested
        if spreadsheet_clears and not DRY_RUN and CLEAR_SPREADSHEET_IDS:
            print("\n" + "=" * 60)
            print("Clearing DDB IDs from Google Sheets...")
            print("=" * 60)
            try:
                update_results = fantasy_sheets.batch_update_cells_by_row_match(SPELLS_GID, "Spell Name", spreadsheet_clears)
                success_count = sum(1 for v in update_results.values() if v)
                print(f"✓ Cleared {success_count}/{len(spreadsheet_clears)} DDB IDs from spreadsheet")
            except Exception as e:
                print(f"✗ Error clearing spreadsheet IDs: {e}")
        
        print()
        print("=" * 60)
        print("DELETE COMPLETE")
        print("=" * 60)
        if DRY_RUN:
            print(f"DRY RUN: Would delete {results['deleted']} spells")
        else:
            print(f"✓ Deleted: {results['deleted']}")
            print(f"✗ Errors: {results['errors']}")
        print("=" * 60)
        
        # Save log
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_file = f"delete_log_{timestamp}.json"
        with open(log_file, 'w') as f:
            json.dump({"timestamp": timestamp, "dry_run": DRY_RUN, "total": len(spells_to_delete), "deleted": results["deleted"], "errors": results["errors"]}, f, indent=2)
        print(f"\n✓ Log saved to: {log_file}")


## 8. Batch Publish Spells

**Publish or unpublish spells on D&D Beyond**

Use this section to batch publish (make public) or unpublish (make private/draft) your homebrew spells.

**Publishing States:**
- **Draft (Status=2)**: Private, only visible to you (default for new spells)
- **Published (Status=1)**: Public, visible in homebrew listings, can be added by others

**How it works:**
1. Fetches all spells with DDB IDs from spreadsheet
2. Optionally fetches current publish status from D&D Beyond
3. Applies publish filter to select spells
4. Publishes or unpublishes selected spells
5. Updates "DDB Publish Status" column in spreadsheet

**Configuration:**
- `ACTION`: Set to `"publish"` or `"unpublish"`
- `DRY_RUN`: Set to `False` to actually change publish status (always test with `True` first!)
- `PUBLISH_FILTER`: Function to filter which spells to publish/unpublish
- `CHECK_CURRENT_STATUS`: Set to `True` to fetch current status from D&D Beyond first (slower)

**⚠️ Important Notes:**
- Published spells are publicly visible
- Once published and approved by moderators, they may be harder to unpublish
- Always test spells as drafts before publishing

In [ ]:
# ============================================
# BATCH PUBLISH/UNPUBLISH SPELLS
# ============================================

ACTION = "publish"  # Set to "publish" or "unpublish"
DRY_RUN = True  # ALWAYS test with True first!
DELAY = 1  # Seconds between requests
VERBOSE = True  # Detailed logging
CHECK_CURRENT_STATUS = False  # Set to True to fetch current status first (slower)

# Define a filter function to select which spells to publish/unpublish
# Examples:
#   - Publish all spells: lambda spell: True
#   - Publish specific spells: lambda spell: spell["name"] in ["Fireball", "Magic Missile"]
#   - Publish by version: lambda spell: spell.get("version", "").startswith("1.")
#   - Unpublish drafts: lambda spell: spell.get("status") == 2

PUBLISH_FILTER = lambda spell: True  # Default: publish nothing (safe)

# Example filters (uncomment one to use):
# PUBLISH_FILTER = lambda spell: spell["name"] in ["Fireball", "Magic Missile"]
# PUBLISH_FILTER = lambda spell: True  # Publish ALL spells

print("=" * 60)
print(f"⚠️  BATCH {ACTION.upper()} SPELLS")
print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No changes will be made")
else:
    print(f"⚠️  LIVE MODE - Spells WILL BE {ACTION.upper()}ED!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Action: {ACTION}")
print(f"Delay between requests: {DELAY}s")
print(f"Check current status: {'YES' if CHECK_CURRENT_STATUS else 'NO'}")
print()

# Ensure publish status column exists
print("Ensuring 'DDB Publish Status' column exists...")
try:
    fantasy_sheets.ensure_column_exists(SPELLS_GID, "DDB Publish Status")
    print("✓ 'DDB Publish Status' column ready")
except Exception as e:
    print(f"⚠️  Could not ensure column: {e}")
print()

# Get all spells with DDB IDs from spreadsheet
print("Loading spells from spreadsheet...")
df_spells = fantasy_sheets.get_sheet(SPELLS_GID)

spell_map = {}
for _, row in df_spells.iterrows():
    spell_name = row.get('Spell Name')
    ddb_id = normalize_ddb_id(row.get('DDB'))
    version = str(row.get('DDB Version', '')).strip()
    status_text = str(row.get('DDB Publish Status', '')).strip().lower()
    
    # Parse status from text
    if status_text in ['published', '1']:
        status = 1
    elif status_text in ['draft', '2']:
        status = 2
    else:
        status = None
    
    if spell_name and ddb_id:
        spell_map[spell_name] = {
            'id': ddb_id,
            'name': spell_name,
            'version': version,
            'status': status
        }

print(f"✓ Found {len(spell_map)} spells with DDB IDs")
print()

# Optionally fetch current status from D&D Beyond
if CHECK_CURRENT_STATUS:
    print("Fetching current publish status from D&D Beyond...")
    print("⚠️  This will be slow (one request per spell)")
    
    for i, (spell_name, spell_info) in enumerate(spell_map.items(), 1):
        if i % 10 == 0:
            print(f"  [{i}/{len(spell_map)}] Checked {i} spells...")
        
        try:
            spell_id = spell_info['id']
            slug = DnDBeyondAPI.create_slug(spell_name)
            details = ddb_api.get_spell_details(spell_id, slug)
            
            if details:
                spell_info['version'] = details.get('version', spell_info['version'])
                spell_info['status'] = details.get('status', spell_info['status'])
            
            time.sleep(0.5)  # Rate limiting
        except Exception as e:
            if VERBOSE:
                print(f"    Warning: Could not fetch status for {spell_name}: {e}")
    
    print(f"✓ Updated status for {len(spell_map)} spells")
    print()

# Apply filter to select spells
spells_to_process = [s for s in spell_map.values() if PUBLISH_FILTER(s)]

print(f"Filter selected {len(spells_to_process)} spells to {ACTION}:")
print()

if len(spells_to_process) == 0:
    print("✓ No spells selected")
    print("  Update PUBLISH_FILTER to select spells")
else:
    for i, spell in enumerate(spells_to_process, 1):
        status_text = "Published" if spell.get('status') == 1 else "Draft" if spell.get('status') == 2 else "Unknown"
        version_text = spell.get('version', 'N/A')
        print(f"{i:3d}. {spell['name']:50s} (ID: {spell['id']}, Status: {status_text}, Ver: {version_text})")
    print()
    
    # Safety check
    if not DRY_RUN:
        print(f"⚠️  WARNING: You are about to {ACTION.upper()} these spells!")
        print()
    
    results = {
        "published": 0,
        "unpublished": 0,
        "errors": 0,
        "details": []
    }
    
    # Track status updates for spreadsheet
    status_updates = []
    
    for i, spell in enumerate(spells_to_process, 1):
        spell_name = spell['name']
        spell_id = spell['id']
        print(f"[{i}/{len(spells_to_process)}] Processing: {spell_name} (ID: {spell_id})")
        
        if DRY_RUN:
            print(f"  → DRY RUN: Would {ACTION} spell")
            if ACTION == "publish":
                results["published"] += 1
                status_updates.append({
                    'match_value': spell_name,
                    'update_column': 'DDB Publish Status',
                    'update_value': 'Published'
                })
            else:
                results["unpublished"] += 1
                status_updates.append({
                    'match_value': spell_name,
                    'update_column': 'DDB Publish Status',
                    'update_value': 'Draft'
                })
        else:
            try:
                if VERBOSE:
                    print(f"  → {ACTION.capitalize()}ing spell...")
                
                if ACTION == "publish":
                    success = ddb_api.publish_spell(spell_id)
                    new_status = "Published"
                elif ACTION == "unpublish":
                    success = ddb_api.unpublish_spell(spell_id)
                    new_status = "Draft"
                else:
                    print(f"  ✗ Invalid action: {ACTION}")
                    results["errors"] += 1
                    continue
                
                if success:
                    print(f"  ✓ {ACTION.capitalize()}ed successfully")
                    if ACTION == "publish":
                        results["published"] += 1
                    else:
                        results["unpublished"] += 1
                    
                    status_updates.append({
                        'match_value': spell_name,
                        'update_column': 'DDB Publish Status',
                        'update_value': new_status
                    })
                else:
                    print(f"  ✗ Failed to {ACTION}")
                    results["errors"] += 1
                
                if i < len(spells_to_process):
                    time.sleep(DELAY)
            except Exception as e:
                print(f"  ✗ Unexpected error: {e}")
                results["errors"] += 1
    
    # Update spreadsheet with new publish status
    if status_updates and not DRY_RUN:
        print("\n" + "=" * 60)
        print("Updating publish status in Google Sheets...")
        print("=" * 60)
        try:
            update_results = fantasy_sheets.batch_update_cells_by_row_match(
                SPELLS_GID,
                "Spell Name",
                status_updates
            )
            success_count = sum(1 for v in update_results.values() if v)
            print(f"✓ Updated {success_count}/{len(status_updates)} spells in spreadsheet")
        except Exception as e:
            print(f"✗ Error updating spreadsheet: {e}")
    
    print()
    print("=" * 60)
    print(f"{ACTION.upper()} COMPLETE")
    print("=" * 60)
    if DRY_RUN:
        print(f"DRY RUN: Would {ACTION} {len(spells_to_process)} spells")
    else:
        print(f"✓ Published: {results['published']}")
        print(f"✓ Unpublished: {results['unpublished']}")
        print(f"✗ Errors: {results['errors']}")
    print("=" * 60)
    
    # Save log
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"{ACTION}_log_{timestamp}.json"
    with open(log_file, 'w') as f:
        json.dump({
            "timestamp": timestamp,
            "action": ACTION,
            "dry_run": DRY_RUN,
            "total": len(spells_to_process),
            "published": results["published"],
            "unpublished": results["unpublished"],
            "errors": results["errors"]
        }, f, indent=2)
    print(f"\n✓ Log saved to: {log_file}")